In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('nitrous_oxide_production')

In [4]:
bd.databases

Databases dictionary with 10 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050
	nitrous-oxide-ei310-Base-2020
	nitrous-oxide-ei310-Base-2050
	nitrous-oxide-ei310-RCP19-2050
	nitrous-oxide-ei310-RCP26-2050

In [5]:
# db = bd.Database('ecoinvent-3.10-cutoff')

In [6]:
# act = [ds for ds in db if 'market for palladium' == ds['name']][0]
# act

In [7]:
# act['reference product']

In [8]:
# ei_db = wurst.extract_brightway2_databases('ecoinvent-3.10-cutoff-Base-2020')

In [9]:
# ei_db = bd.Database('ecoinvent-3.10-cutoff-Base-2020')

In [10]:
# for ds in ei_db:
#     if 'market for hydrogen, gaseous, medium pressure, merchant' in ds['name'] and 'kilogram' in ds['unit']:
#         print(ds['reference product'], ds['name'], ds['location'], ds['unit'])
#         act = ds

In [11]:
# lca = bc.LCA({act: 1}, method=("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2"))
# lca.lci()
# lca.lcia()
# lca.score

In [12]:
# import inventories from excel
excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories_ei310.xlsx'))
# excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories.xlsx'))
excel_import.apply_strategies()
excel_import.match_database('ecoinvent-3.10-cutoff-Base-2020', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-biosphere', fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excel_import.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from new database
excel_import.statistics()

Extracted 5 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 10.93 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
21 datasets
	191 exchanges
	Links to the fo

(21, 191, 0, 0)

In [13]:
excel_import.write_database()

100%|██████████| 21/21 [00:00<00:00, 595.42it/s]


10:30:57 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-Base-2020


In [20]:
act = [ds for ds in bd.Database('ecoinvent-3.10-cutoff-Base-2020') if 'phenol production, cumene oxidation' in ds['name'] and 'kilogram' in ds['unit']][0]
act

'phenol production, cumene oxidation' (kilogram, RER, None)

In [24]:
act['reference product']

'phenol'

In [21]:
for exc in act.exchanges():
    print(exc)

Exchange: 1.0 kilogram 'phenol production, cumene oxidation' (kilogram, RER, None) to 'phenol production, cumene oxidation' (kilogram, RER, None)>
Exchange: 5.19022551695778e-10 unit 'chemical factory construction, organics' (unit, RER, None) to 'phenol production, cumene oxidation' (kilogram, RER, None)>
Exchange: 1.0840809959558086 kilogram 'market for cumene' (kilogram, GLO, None) to 'phenol production, cumene oxidation' (kilogram, RER, None)>
Exchange: 0.00040119478662605396 kilogram 'market for sulfuric acid' (kilogram, RER, None) to 'phenol production, cumene oxidation' (kilogram, RER, None)>
Exchange: -1.7822504661804228e-06 cubic meter 'market for wastewater, average' (cubic meter, Europe without Switzerland, None) to 'phenol production, cumene oxidation' (kilogram, RER, None)>
Exchange: -4.4790592114626854e-08 cubic meter 'market for wastewater, average' (cubic meter, CH, None) to 'phenol production, cumene oxidation' (kilogram, RER, None)>
Exchange: 6.475600055923602e-05 kilo

In [22]:
method = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

In [23]:
lca = bc.LCA({act: 1}, method = method)
lca.lci()
lca.lcia()
lca.score

4.449040338457731